# Tutorial 1 — The "advil moment"

**Find hidden synonyms with one search query.**

In this 20-minute hands-on, you'll watch a search for `advil` surface `motrin`, `aleve`, and `ibuprofen` — without anyone telling the system those terms are related. No hand-curated synonym list. No trained model. No embeddings.

Just your inverted index, used cleverly.

This is the central trick of Chapter 5 of *AI-Powered Search* — the **Semantic Knowledge Graph** (SKG, §5.4 in the book). By the end of this notebook you'll be able to:

1. Run an SKG traversal against any term in your corpus.
2. Read the JSON facet wire format and explain it line-by-line.
3. Visualize the resulting semantic neighborhood as a graph.
4. Explain why this works while a naive `terms` facet does not.

> **Who this is for.** This tutorial assumes you've worked with an inverted-index search engine before (Solr, Elasticsearch, OpenSearch). You don't need to have read every page of Chapter 5 — we'll re-derive the bits you need.

## Prerequisites

You should already have the book's standard setup:

1. Cloned the [`ai-powered-search` repo](https://github.com/ai-powered-search).
2. Run the Docker stack (`docker compose up`) so Solr is reachable on `localhost:8983`.
3. Indexed the Stack Exchange **health** collection (see appendix A of the book, or the chapter-5 setup notebook).

Quick sanity check — the next cell should print one document's title.

In [ ]:
from aips import get_engine

engine = get_engine()
health = engine.get_collection("health")

# A plain keyword search. If this works, the rest of the tutorial will too.
result = health.search(query="advil", limit=1, return_fields=["title"])
if result["docs"]:
    print(f"Sample doc: {result['docs'][0]['title']}")
else:
    print("⚠️  No docs returned. Is the 'health' collection indexed?")

## The problem we're solving

Keyword search has a literal-spelling problem.

A user who types `advil` matches documents containing the string `advil`. A user who types `motrin` matches documents containing the string `motrin`. Even though those drugs are interchangeable to a doctor — and to many users — the engine treats them as unrelated character sequences.

The traditional fix is a **synonyms file**: a hand-maintained list like `advil, motrin, ibuprofen => advil`. That works, but:

- Someone has to curate it.
- It doesn't reflect *your* corpus; it reflects whoever wrote the list.
- It can't keep up with a domain that changes (new drugs, new slang).
- It's lossy — it collapses meaning rather than measuring relatedness.

**What if the corpus could tell us, in real time, which terms are semantically related — for free?**

## A first try: just facet on the term field

If we want to know which terms appear alongside `advil`, we could simply search for `advil` and ask Solr to return the most common terms in the matching documents' `body` field. That's a standard `terms` facet — you've written it a hundred times.

In [ ]:
naive_request = {
    "query": "advil",
    "limit": 0,
    "facet": {
        "top_body_terms": {
            "type": "terms",
            "field": "body",
            "limit": 10,
            "mincount": 2,
        }
    },
}

response = health.native_search(naive_request)

print(f"{'TERM':<20} {'COUNT'}")
print("-" * 30)
for bucket in response["facets"]["top_body_terms"]["buckets"]:
    print(f"{bucket['val']:<20} {bucket['count']}")

## Why that didn't work

You probably saw `the`, `a`, `of`, `to`, `is`, `i`, `and`... in the top spots. Garbage.

The problem isn't that those terms appear *with* advil — they do. It's that they appear *with everything*. High raw co-occurrence is a noisy signal. What we actually want to ask is:

> Does this term appear with `advil` **more** than it appears with random other words?

That's the *distributional hypothesis* (§5.4.4): words with similar meanings appear in similar distributions. So we need to compare two distributions — the **foreground** (docs matching `advil`) against a **background** (the corpus as a whole). A term is semantically related to `advil` if it's *over-represented* in the foreground vs. the background.

That comparison is the **relatedness** function from §5.4.4:

$$
\mathrm{relatedness}(x, fg, bg) = \frac{|D_x \cap D_{fg}| - |D_{fg}| \cdot P_x}{\sqrt{|D_{fg}| \cdot P_x \cdot (1 - P_x)}}
$$

where $|D_x \cap D_{fg}|$ is how many foreground docs contain candidate term $x$, $|D_{fg}|$ is the size of the foreground set, and $P_x$ is the probability of $x$ appearing in a random background doc. The result gets sigmoid-normalized to land in $[-1, 1]$.

Conceptually: it's a **z-score** for *"how surprising is it that this term shows up here?"* — with stopwords getting scores near zero because they're unsurprising everywhere.

## The fix: relatedness as a faceting aggregation

Solr has the relatedness calculation built into its JSON faceting API (`RelatednessAgg`). The request looks almost identical to the naive one above — we're just sorting by `relatedness` instead of by `count`, and we're declaring a **fore**ground (docs matching `advil`) and a **back**ground (`*:*`, all docs).

Here's the raw wire format. Read it once and convince yourself it's not magic — it's a faceting query.

In [ ]:
import json

skg_request = {
    "query": "*:*",                # match everything; we narrow with $fore
    "limit": 0,                    # don't return docs, only the facets
    "params": {
        "fore": "{!${defType} v=$f0_0_query}",
        "back": "*:*",
        "defType": "edismax",
        "f0_0_query": "advil",     # foreground = docs matching advil
    },
    "facet": {
        "related_terms": {
            "type": "terms",
            "field": "body",
            "mincount": 2,
            "limit": 8,
            "sort": {"relatedness": "desc"},
            "facet": {
                "relatedness": {
                    "type": "func",
                    "func": "relatedness($fore,$back)",
                },
            },
        }
    },
}

print(json.dumps(skg_request, indent=2))

That's the wire format. You can paste it into the Solr admin UI and run it yourself if you want to debug. But the book's `aips` package gives a cleaner abstraction — the *traversal* API — which is what we'll use for the rest of the tutorial.

In [ ]:
from aips import get_skg

health_skg = get_skg(health)

# The traversal reads like a sentence:
#   "Starting from the term 'advil' in the body field, find the top 8 terms
#    in the body field that appear together with it at least twice."
nodes_to_traverse = [
    {"field": "body", "values": ["advil"]},
    {"field": "body", "min_occurrences": 2, "limit": 8},
]

traversal = health_skg.traverse(*nodes_to_traverse)

In [ ]:
from aips.helpers.print_utils import print_graph

print_graph(traversal, "advil")

## 🎯 The moment

The output ranks terms by relatedness to `advil` **within the health corpus**:

```
advil          0.71
motrin         0.60
aleve          0.47
ibuprofen      0.38
alleve         0.37
tylenol        0.33
naproxen       0.31
acetaminophen  0.18
```

Read that list. We just discovered, automatically, that:

- `motrin`, `aleve`, `ibuprofen`, and `naproxen` are siblings of `advil` (all NSAIDs).
- `tylenol` and `acetaminophen` are related — but with *lower* scores (different drug class, not direct substitutes).
- `alleve` (a misspelling) was caught too — relatedness picks up on conceptually-similar character sequences whether they're spelled correctly or not.

No synonym list. No trained embedding. No `fine_tune()`. Just two index lookups and a z-score.

> **The "find itself" effect.** Notice the query term `advil` shows up with the highest score (~0.71). That's expected — the foreground set is *defined by* containing `advil`, so `advil` co-occurs with itself in 100% of foreground docs. Useful sanity check: if your top-ranked term isn't the query itself, something's off.

## Visualize it

Two views, both in the helper module `skg_viz.py` (sitting alongside this notebook):

1. A horizontal bar chart of relatedness scores (good for ranking).
2. A force-directed graph centered on the query (good for showing structure).

In [ ]:
from skg_viz import plot_relatedness_bars

plot_relatedness_bars(traversal, "advil")

In [ ]:
from skg_viz import draw_graph

draw_graph(traversal, "advil")

## What just happened, in IR terms

For each traversal step, the engine does exactly two index operations:

1. **Inverted index lookup** — *which documents contain `advil`?* That set is the **foreground**.
2. **Forward index (`docValues`) lookup** — *which terms appear in those documents?* Those are the **candidate neighbors**.

Then for each candidate term $x$, it computes `relatedness(x, fore, back)` using the formula above. The candidates with the highest z-scores are the "semantic neighbors" of `advil`.

That's it. Not magic — the same primitives you use every day, composed in one specific way:

| Operation                          | Index used        |
|------------------------------------|-------------------|
| "What docs match this query?"      | Inverted index    |
| "What terms are in this doc set?"  | Forward index     |
| "How surprising is each term?"     | `relatedness()`   |

Crucially, the SKG is **materialized on the fly**. There's no graph stored anywhere on disk. Indexing a new document doesn't trigger a graph update — the graph is *implied* by the index and traversed at query time. §5.4.3 has the data-structure diagrams; figure 5.6 (p. 117) is the clearest picture.

## Your turn — hands on

Try traversing other starting nodes. Suggestions in this dataset:

- `headache` — what kind of neighbors? Symptoms, drugs, or both?
- `pregnancy` — does the SKG cluster medical terminology or lifestyle terms?
- `knee pain` — multi-word starting node; how does it compare to single words?
- A noisy word like `feel` — watch the relatedness collapse the scores.

Two knobs to play with:

- `min_occurrences` — filter out rare terms. Default 2. Raise it to 5 or 10 on a noisy field and the graph cleans up.
- `limit` — how many neighbors to return. Bigger = more recall, more noise.

In [ ]:
# ── Try changing these ─────────────────────────────────────────────
my_query = "headache"        # ← swap me
min_occurrences = 2          # ← raise this to reduce noise
limit = 10                   # ← how many neighbors to surface
# ─────────────────────────────────────────────────────────────

nodes_to_traverse = [
    {"field": "body", "values": [my_query]},
    {"field": "body", "min_occurrences": min_occurrences, "limit": limit},
]
my_traversal = health_skg.traverse(*nodes_to_traverse)

print_graph(my_traversal, my_query)
plot_relatedness_bars(my_traversal, my_query)
draw_graph(my_traversal, my_query)

## Discussion — things worth pausing on

1. **Why is the query itself the top result?** The foreground is *defined by* matching the query, so the query trivially co-occurs with itself in 100% of foreground docs. Its score is therefore the highest. This is a feature — it gives you a built-in upper bound on relatedness, useful when comparing strengths across queries.

2. **What corpora does this fail on?** If every document covers a unique topic with no shared vocabulary, the foreground/background distinction collapses and scores stay near zero. This is why the book recommends Stack Exchange forums (lots of overlap) and warns against Wikipedia (one canonical doc per topic — see §5.4.2).

3. **What's the production cost?** The traversal is one Solr request. Latency is comparable to a regular facet query (tens of ms on this dataset). It's cheap enough to call per-query — which is what Tutorial 2 exploits when we use the neighbors to expand the search query.

4. **What about phrases?** The starting node can be any query — `knee pain`, `"chronic fatigue"`, even a Boolean. Internally it's resolved to a doc set the same way as a single term.

## What's next

You now have the core SKG primitive. The next tutorials in this series show what you can *do* with it:

- **Tutorial 2 — Query expansion that moves the needle.** Take a sparse query (like `vibranium`) and use the SKG-derived neighbors to rewrite it. Compare four expansion strategies (simple OR, min-match threshold, AND-with-boosts, boost-only re-ranking) on the same query, side by side.

- **Tutorial 3 — Beyond "is related to".** Use an intermediate "relationship node" (e.g., `in love with`) to materialize a typed edge — the X-Men example from §5.4.7, built from the same traversal primitive.

- **Tutorial 4 — Compare with explicit knowledge graphs.** Run Hearst patterns over the same dataset to extract `is_a` triples, then compare against what the SKG gives dynamically. When to use which?

For now, close the notebook, restart your kernel, and re-run it with a different starting term. The "click" usually happens on the second or third example.